# From State Space Search to Mamba

These notes preserve the full didactic explanations developed throughout the conversation.

# Chapter 1 — What is a State?


Before talking about State Space Models, we must answer a more fundamental question:

> **What is a state?**

A **state** is all the information necessary to completely describe a system at a given instant.

For the 8-puzzle, the state is **the entire board configuration**, not the position of a single tile.

Example:

```text
1 2 3
4 5 6
7 _ 8
```

Knowing only that *tile 5 is in the center* is insufficient because many different boards satisfy that condition.

Therefore, the state must contain **all the information required** to uniquely describe the current situation.

A state may be represented in different ways, for example

$$
s=
\begin{bmatrix}
2&8&3\\
1&6&4\\
7&\square&5
\end{bmatrix}
$$

or equivalently

$$
s=(2,8,3,1,6,4,7,0,5).
$$

The important idea is not the representation itself, but that the state completely describes the system.


# Chapter 2 — State Transition


A state alone does nothing.

The interesting question is:

> **How does the system move from one state to another?**

If

```text
2 8 3
1 6 4
7 _ 5
```

becomes

```text
2 8 3
1 6 4
7 5 _
```

something produced that change.

That mechanism is called the **transition function**.

Mathematically,

$$
s_{t+1}=T(s_t,a_t)
$$

where

- $s_t$ is the current state,
- $a_t$ is an action,
- $T$ is the transition function.

At this point we are still thinking like classical AI:

Current State → Action → Next State.


# Chapter 3 — From State to Memory


Dynamic systems ask a different question.

Instead of searching for a solution, they ask:

> **What information is required to predict the future?**

This leads to the **Markov Property**:

> The future depends only on the current state, not on the complete history.

For physical systems we replace the board configuration with a state vector

$$
x_t
$$

and the action with an external input

$$
u_t.
$$

The transition becomes

$$
x_{t+1}=f(x_t,u_t).
$$

The state is now interpreted as a **compact memory** containing everything required to predict the future.


# Chapter 4 — State Equations


The function

$$
x_{t+1}=f(x_t,u_t)
$$

is initially a black box.

It receives

- the current state,
- the current input,

and produces the next state.

At this point we do not yet know how this function is computed.


# Chapter 5 — Linear State Space Models


Suppose the dynamics can be approximated by a linear function.

Then

$$
f(x_t,u_t)=Ax_t+Bu_t
$$

and therefore

$$
x_{t+1}=Ax_t+Bu_t.
$$

A useful intuition is that this equation is simply the matrix generalization of a linear function.

Classical linear equation:

$$
y=mx+b.
$$

State equation:

$$
x_{t+1}=Ax_t+Bu_t.
$$

Clarifications:

- $A$ describes how the state evolves by itself.
- $B$ describes how the external input affects the state.

A more general affine model could be written as

$$
x_{t+1}=Ax_t+Bu_t+c,
$$

but in control theory the origin is usually shifted to an equilibrium point, eliminating the constant term.

This was one of the conceptual questions discussed during the lesson.


# Chapter 6 — Continuous Systems


Physical systems evolve continuously.

Instead of computing the next state,

we compute **how the state changes**:

$$
\dot{x}(t)=Ax(t)+Bu(t).
$$

Important clarification:

This equation **does not return the next state**.

It returns the **rate of change** of the state.

To recover the state we must integrate:

$$
x(t)=x(t_0)+\int_{t_0}^{t}\dot{x}(\tau)d\tau.
$$

Therefore,

- Discrete equation → directly computes the next state.
- Continuous equation → computes the derivative.
- Integration recovers the state.

This distinction is fundamental before studying S4.


# Chapter 7 — Discretization


Computers process discrete samples.

Approximating

$$
\dot{x}(t)\approx\frac{x_{t+1}-x_t}{\Delta}
$$

gives Euler discretization

$$
x_{t+1}=x_t+\Delta(Ax_t+Bu_t).
$$

Interpretation:

Next State = Current State + Small Change.


# Chapter 8 — Exact Discretization


Euler accumulates approximation error.

Instead, State Space Models use the exact solution of the linear differential equation:

$$
x_{k+1}=\bar A x_k+\bar B u_k.
$$

where

$$
\bar A=e^{A\Delta},
$$

and

$$
\bar B=\left(\int_0^\Delta e^{A\tau}d\tau\right)B.
$$

This preserves the continuous dynamics much more faithfully than Euler.


# Chapter 9 — Hidden State


The state is reinterpreted once again.

Instead of representing position and velocity,

it becomes

> **a compressed memory of everything observed so far.**

Every new token or frame updates that memory through

$$
x_{t+1}=\bar A x_t+\bar B u_t.
$$

The mathematics remains the same.

Only the interpretation changes.


# Chapter 10 — Why RNNs Were Not Enough?


RNNs also maintain memory,

$$
h_t=\phi(W_hh_{t-1}+W_xx_t+b),
$$

but suffer from

- vanishing gradients,
- sequential computation.

Transformers solved long-range dependencies with attention, but require

$$
O(n^2)
$$

operations.


# Chapter 11 — The Birth of S4


S4 (Structured State Space for Sequence Modeling) asked a simple question:

> Can classical Linear State Space Models become sequence models?

Its main innovation was showing that State Space Models could efficiently process very long sequences.

However,

the dynamics were **fixed**.

Every token used the same matrices

$$
\bar A,\;\bar B.
$$


# Chapter 12 — Mamba


Mamba addresses S4's main limitation.

Instead of fixed dynamics,

the update becomes conceptually

$$
x_{k+1}=\bar A(u_k)x_k+\bar B(u_k)u_k.
$$

The matrices depend on the current input.

Therefore,

the hidden state becomes **adaptive** instead of passive.

Final historical evolution:

RNN → LSTM → Transformer → S4 → Mamba.

The key conceptual takeaway from the entire notebook is:

- Classical AI: state = configuration.
- Control Theory: state = sufficient information to predict the future.
- S4: state = memory of a sequence.
- Mamba: state = adaptive memory whose update depends on the current input.
